[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-0/basics.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/56295530-getting-set-up-video-guide)

# LangChain Academy

Welcome to LangChain Academy! 

## Context

At LangChain, we aim to make it easy to build LLM applications. One type of LLM application you can build is an agent. There’s a lot of excitement around building agents because they can automate a wide range of tasks that were previously impossible. 

In practice though, it is incredibly difficult to build systems that reliably execute on these tasks. As we’ve worked with our users to put agents into production, we’ve learned that more control is often necessary. You might need an agent to always call a specific tool first or use different prompts based on its state. 

To tackle this problem, we’ve built [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) — a framework for building agent and multi-agent applications. Separate from the LangChain package, LangGraph’s core design philosophy is to help developers add better precision and control into agent workflows, suitable for the complexity of real-world systems.

## Course Structure

The course is structured as a set of modules, with each module focused on a particular theme related to LangGraph. You will see a folder for each module, which contains a series of notebooks. A video will accompany each notebook to help walk through the concepts, but the notebooks are also stand-alone, meaning that they contain explanations and can be viewed independently of the videos. Each module folder also contains a `studio` folder, which contains a set of graphs that can be loaded into [LangSmith Studio](https://docs.langchain.com/langsmith/quick-start-studio), our IDE for building LangGraph applications.

## Setup

Before you begin, please follow the instructions in the `README` to create an environment and install dependencies.

## Chat models

In this course, we'll use Chat Models, which take a sequence of messages as input and return messages as output. LangChain supports many models via [third-party integrations](https://docs.langchain.com/oss/python/integrations/chat). By default, the course will use  [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai) because it is both popular and performant. As noted, please ensure that you have an `OPENAI_API_KEY`.

Let's check that your `OPENAI_API_KEY` is set and, if not, you will be asked to enter it.

In [31]:
import jupyter_client
jupyter_client.find_connection_file()


# List all kernel specs
ksm = jupyter_client.kernelspec.KernelSpecManager()
print(ksm.get_all_specs())

import os
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "jupyter", "jupyter_client", "jupyter_core", "notebook", "langchain_openai", "langchain_core", "langchain_community", "langgraph-api", "langchain-tavily"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-U", "querychat @ git+https://github.com/ruralinnovation/querychat"], check=True)


{'python3': {'resource_dir': '/Users/johnhall/Documents/edu/LangGraph/.venv/share/jupyter/kernels/python3', 'spec': {'argv': ['python', '-m', 'ipykernel_launcher', '-f', '{connection_file}'], 'env': {}, 'display_name': 'Python 3 (ipykernel)', 'language': 'python', 'interrupt_mode': 'signal', 'metadata': {'debugger': True}}}, 'langgraph': {'resource_dir': '/Users/johnhall/Library/Jupyter/kernels/langgraph', 'spec': {'argv': ['/Users/johnhall/Documents/edu/LangGraph/.venv/bin/python', '-Xfrozen_modules=off', '-m', 'ipykernel_launcher', '-f', '{connection_file}'], 'env': {'PATH': '/Users/johnhall/Documents/edu/LangGraph/.venv/bin:/usr/local/bin:/usr/bin:/bin:/usr/sbin:/sbin'}, 'display_name': 'Python 3 (ipykernel)', 'language': 'python', 'interrupt_mode': 'signal', 'metadata': {'debugger': True}}}}


CompletedProcess(args=['/Users/johnhall/Documents/edu/LangGraph/.venv/bin/python', '-m', 'pip', 'install', '--quiet', '-U', 'querychat @ git+https://github.com/ruralinnovation/querychat'], returncode=0)

In [2]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

if "OPENAI_API_KEY" not in os.environ:
    _set_env("OPENAI_API_KEY")

if "OPENROUTER_API_KEY" not in os.environ:
    _set_env("OPENROUTER_API_KEY")

[Here](https://docs.langchain.com/oss/python/langchain/models) is a useful how-to for all the things that you can do with chat models, but we'll show a few highlights below. If you've run `pip install -r requirements.txt` as noted in the README, then you've installed the `langchain-openai` package. With this, we can instantiate our `ChatOpenAI` model object. You can see pricing for various models [here](https://openai.com/api/pricing/). The notebooks will default to `gpt-4o` because it offers a good balance of quality, price, and speed, but you can also opt for the lower-priced `gpt-3.5` series or more recent models.

There are [a few standard parameters](https://docs.langchain.com/oss/python/langchain/models#parameters) that we can set with chat models. Two of the most common are:

* `model`: the name of the model
* `temperature`: the sampling temperature

`Temperature` controls the randomness or creativity of the model's output where low temperature (close to 0) is more deterministic and focused outputs. This is good for tasks requiring accuracy or factual responses. High temperature (close to 1) is good for creative tasks or generating varied responses. 

In [16]:
from langchain_openai import ChatOpenAI
gpt4o_chat = ChatOpenAI(model="gpt-4o", temperature=0)
gpt35_chat = ChatOpenAI(model="gpt-3.5-turbo-0125", temperature=0)
gpt_oss_chat = ChatOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    # model="openai/gpt-oss-120b:free",   # OpenAI: gpt-oss-120b (free)
                                           # Free Users with $10+ in credits
                                           # - 1,000 requests per day
                                           # - 20 requests per minute
    model="openai/gpt-oss-120b",           # OpenAI: gpt-oss-120b (paid)
                                           # $0.039/M input, $0.19/M output
    temperature=0,
    default_headers={
        "HTTP-Referer": "localhost",  # Optional. Site URL for rankings on openrouter.ai.
        "X-Title": "langchain-academy",  # Optional. Site title for rankings on openrouter.ai.
    },
    extra_body={
        "provider": {
            "sort": "latency",           # prioritize lowest latency providers
            "allow_fallbacks": True,     # fall back to others if unavailable
        }
    }
)

Chat models in LangChain have a number of [default methods](https://reference.langchain.com/python/langchain_core/runnables). For the most part, we'll be using:

* [stream](https://docs.langchain.com/oss/python/langchain/models#stream): stream back chunks of the response
* [invoke](https://docs.langchain.com/oss/python/langchain/models#invoke): call the chain on an input

And, as mentioned, chat models take [messages](https://docs.langchain.com/oss/python/langchain/messages) as input. Messages have a role (that describes who is saying the message) and a content property. We'll be talking a lot more about this later, but here let's just show the basics.

In [27]:
from langchain_core.messages import HumanMessage

# Create a message
msg = HumanMessage(content="Hello world", name="Lance")

# Message list
messages = [msg]

# Invoke the model with a list of messages 
# gpt4o_chat.invoke(messages)
gpt_oss_chat.invoke(messages)

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 80, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 2.55e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.55e-05, 'upstream_inference_prompt_cost': 8e-06, 'upstream_inference_completions_cost': 1.75e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'gen-1770214397-dmX04snAWctbS7nG3vNB', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c2900-1418-7712-9efe-39bd2d1f7e16-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 35, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

We get an `AIMessage` response. Also, note that we can just invoke a chat model with a string. When a string is passed in as input, it is converted to a `HumanMessage` and then passed to the underlying model.


In [28]:
# gpt4o_chat.invoke("hello world")
gpt_oss_chat.invoke("hello world")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 80, 'total_tokens': 115, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 20, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'cost': 2.55e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.55e-05, 'upstream_inference_prompt_cost': 8e-06, 'upstream_inference_completions_cost': 1.75e-05}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'gen-1770214400-eq5IIne7HfQyMuGrQNXL', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c2900-20bf-7c62-adb1-dce0150d4773-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 80, 'output_tokens': 35, 'total_tokens': 115, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'outpu

The interface is consistent across all chat models and models are typically initialized once at the start up each notebooks. 

So, you can easily switch between models without changing the downstream code if you have strong preference for another provider.


## Search Tools

You'll also see [Tavily](https://tavily.com/) in the README, which is a search engine optimized for LLMs and RAG, aimed at efficient, quick, and persistent search results. As mentioned, it's easy to sign up and offers a generous free tier. Some lessons (in Module 4) will use Tavily by default but, of course, other search tools can be used if you want to modify the code for yourself.

In [8]:
_set_env("TAVILY_API_KEY")

In [29]:
from langchain_tavily import TavilySearch  # updated at 1.0

tavily_search = TavilySearch(max_results=3)

data = tavily_search.invoke({"query": "What is LangGraph?"})
search_docs = data.get("results", data)
search_docs

[{'url': 'https://www.geeksforgeeks.org/machine-learning/what-is-langgraph/',
  'title': 'What is LangGraph? - GeeksforGeeks',
  'content': 'LangGraph is an open-source framework built by LangChain that streamlines the creation and management of AI agent workflows.',
  'score': 0.9449999,
  'raw_content': None},
 {'url': 'https://www.ibm.com/think/topics/langgraph',
  'title': 'What is LangGraph? - IBM',
  'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows.',
  'score': 0.9438932,
  'raw_content': None},
 {'url': 'https://www.datacamp.com/tutorial/langgraph-tutorial',
  'title': 'LangGraph Tutorial: What Is LangGraph and How to Use It?',
  'content': 'LangGraph is a library within the LangChain ecosystem that provides a framework for defining, coordinating, and executing multiple LLM agents (or chains) in a structured and efficient manner. By managing the flow of data and the seque

In [30]:
data = tavily_search.invoke({"query": "What is Tavily Search?"})
search_docs = data.get("results", data)
search_docs

[{'url': 'https://tavily.com/',
  'title': 'Tavily',
  'content': "# Connect your AI agents to the web. Real-time search, extraction, research, and web crawling through a single, secure API. /the web access layer for agents. ### Ground models with fresh web context. Retrieve live web data, extract relevant content, and return it structured and chunked for models, so agents reason over facts without hallucinating. ### Handle thousands of web queries in seconds. A production-grade retrieval stack with real-time search, intelligent caching, and indexing keeps latency predictable as traffic grows. ### Web Search Driven by Research. This benchmark evaluates factual question answering using OpenAI's SimpleQA, which measures how accurately models answer short, fact-seeking queries. Model: GPT-4.1, grounded by retrieved documents from provider. Retrieval: max 10 documents per query. p50 on Tavily /search making us fastest on the market. ## Tavily in action. ### Databricks Partners with Tavily 